# PolyTutor Fine-Tuning Pipeline
Runs all four training phases end-to-end on Kaggle with A100 GPU.

**Before running:** Add `HF_TOKEN` and `HF_USERNAME` to Kaggle Secrets (Add-ons → Secrets).

In [ ]:
# Cell 1 — Environment setup
!pip install -q -r training/requirements_training.txt
import os
HF_TOKEN    = os.environ["HF_TOKEN"]     # set in Kaggle secrets
HF_USERNAME = os.environ["HF_USERNAME"]  # set in Kaggle secrets
print("Environment ready")

In [ ]:
# Cell 2 — GPU check
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
assert torch.cuda.is_available(), "No GPU — check Kaggle accelerator settings"

In [ ]:
# Cell 3 — Dataset preparation
%run training/01_prepare_dataset.py
# Expected output: summary table showing 11,500+ examples

In [ ]:
# Cell 4 — Fine-tuning
%run training/02_finetune.py
# Expected duration: ~3 hours on A100
# Saves to training/outputs/polytutor-weights/

In [ ]:
# Cell 5 — Evaluation
%run training/03_evaluate.py
# Generates training/outputs/benchmark_report.md

In [ ]:
# Cell 6 — Display report
with open("training/outputs/benchmark_report.md") as f:
    print(f.read())

In [ ]:
# Cell 7 — Upload report to Hugging Face
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="training/outputs/benchmark_report.md",
    path_in_repo="benchmark_report.md",
    repo_id=f"{HF_USERNAME}/polytutor-gemma3-9b",
    token=HF_TOKEN
)
print(f"Report published: https://huggingface.co/{HF_USERNAME}/polytutor-gemma3-9b")